# Stage 3 — Embedding Model Exploration (v2)

**Goal:** decide between three embedding models, with and without LaTeX normalization.

**Models compared:**
1. `paraphrase-multilingual-MiniLM-L12-v2` — current baseline (~500 MB, 384-dim, fast)
2. `intfloat/multilingual-e5-base` — first upgrade (~1 GB, 768-dim, needs `passage:`/`query:` prefixes)
3. `BAAI/bge-m3` — strongest multilingual option (~2.3 GB, 1024-dim, no prefixes)

**LaTeX normalization:** convert `\(\frac{1}{2}\)` → `(1)/(2)`, `\sin(x)` → `sin(x)`, etc., so the embedder sees real words instead of backslash tokens.

---

## Prerequisites

Run in terminal before opening this notebook:
```bash
source venv/bin/activate
pip install pydantic pylatexenc jupyter   # if not already installed
```

**Why `pylatexenc`?** It's a real LaTeX parser (vs our zero-dependency regex fallback). Handles nested structures and edge cases more accurately. Tiny package, pure Python. When installed, `normalize_latex()` automatically uses it.

---

**Memory note:** loading all three models at once needs ~4 GB RAM. If your Mac has only 8 GB, skip the BGE-M3 load (commented marker in cell 7) and run it separately.

**What to look for:**
- Does BGE-M3 retrieve more relevant rows than MiniLM/e5?
- Does LaTeX normalization clearly help any model on math queries?
- Are differences large enough to matter, or marginal?

## 1. Load data and build a stratified sample

In [14]:
import sys
!{sys.executable} -m pip install pylatexenc


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip


In [15]:
import json
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path

# Make project imports work from notebooks/
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.latex import normalize_latex, _HAS_PYLATEX

# Environment sanity check
print(f"Python: {sys.version.split()[0]}")
print(f"LaTeX engine: {'pylatexenc' if _HAS_PYLATEX else 'regex fallback'}")
if not _HAS_PYLATEX:
    print("  (install with: pip install pylatexenc for better coverage)")

READY_PATH = ROOT / "data" / "processed" / "ready.jsonl"
rows = [json.loads(line) for line in open(READY_PATH, encoding="utf-8")]
print(f"\nTotal rows: {len(rows)}")
print(f"By language: {Counter(r['language'] for r in rows)}")

Python: 3.13.0
LaTeX engine: pylatexenc

Total rows: 10608
By language: Counter({'en': 4816, 'fr': 3660, 'ar': 2132})


In [16]:
# Stratified sample: 100 rows per language.
random.seed(42)
by_lang = defaultdict(list)
for r in rows:
    by_lang[r["language"]].append(r)

SAMPLE_PER_LANG = 100
sample = []
for lang, lang_rows in by_lang.items():
    sample.extend(random.sample(lang_rows, min(SAMPLE_PER_LANG, len(lang_rows))))

print(f"Sample size: {len(sample)}  (by lang: {Counter(r['language'] for r in sample)})")

Sample size: 300  (by lang: Counter({'en': 100, 'fr': 100, 'ar': 100}))


## 2. LaTeX normalization preview

Quick sanity check that normalization actually changes the math content as expected.

In [17]:
from src.data.latex import contains_latex, _HAS_PYLATEX

engine_label = "pylatexenc" if _HAS_PYLATEX else "regex fallback"
print(f"Normalizing with: {engine_label}\n")

# question_text is the payload field — it keeps the raw LaTeX even when
# search_text was already normalized at Stage 2c. Use it to preview.
math_samples = [r for r in sample if contains_latex(r["question_text"])][:5]

if not math_samples:
    print("No LaTeX-containing rows in the sample.")
    print("Bump SAMPLE_PER_LANG above from 100 to 200 and re-run.")

for r in math_samples:
    raw = r["question_text"]
    norm = normalize_latex(raw)
    print(f"[{r['language']}]  subjects={r['subjects']}")
    print(f"  RAW question_text   : {raw[:220]}")
    print(f"  NORMALIZED ({engine_label:18s}): {norm[:220]}")
    print(f"  search_text in row  : {r['search_text'][:220]}")
    print()

Normalizing with: pylatexenc

[en]  subjects=[]
  RAW question_text   : \(​Soit\ x\ \epsilon\ \left[0;\pi\right],\cos\left(\pi-x\right)=\) \(a)\ \cos(x)\) \(b)\ \sin(x)\) \(c)\ -\cos(x)\)
  NORMALIZED (pylatexenc        ): ​Soit x ϵ [0;π],cos(π-x)= a) cos(x) b) sin(x) c) -cos(x)
  search_text in row  : trigonométrie 2. ​Soit x ϵ [0;π],cos(π-x)= a) cos(x) b) sin(x) c) -cos(x). a | b | c

[en]  subjects=['MATHEMATICS']
  RAW question_text   : \(\lim_{x\to0^+\ }x\ln\left(\frac{x+1}{x}\right)=\) a) \(0\) b) \(1\) c) \(+\infty\)
  NORMALIZED (pylatexenc        ): lim_x→0^+ xln(x+1/x)= a) 0 b) 1 c) +∞
  search_text in row  : MATHEMATICS. Fonction logharithéme. lim_x→0^+ xln(x+1/x)= a) 0 b) 1 c) +∞. a | b | c

[en]  subjects=['MATHEMATICS']
  RAW question_text   : Pour tout nombre réel \(x\) , \(A(x)=1-\frac{e^{-x}-1}{e^{-x}+1}\) s'écrit également: a) \(2\) b) \(\frac{2}{e^{-x}+1}\) c) \(\frac{2e^{-x}}{e^{-x}+1}\)
  NORMALIZED (pylatexenc        ): Pour tout nombre réel x , A(x)=1-e^-x-1/e^-x

## 3. Load the three models

First run downloads ~4 GB total (~3-5 min on decent internet). Cached afterwards.

In [18]:
from sentence_transformers import SentenceTransformer
import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")

print("\nLoading MiniLM (~500 MB)...")
minilm = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)

print("\nLoading e5-base (~1 GB)...")
e5 = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

# COMMENT THIS OUT IF YOU HAVE <8 GB RAM and want to skip BGE-M3
print("\nLoading BGE-M3 (~2.3 GB)...")
bge = SentenceTransformer("BAAI/bge-m3", device=device)

print(f"\nMiniLM dim: {minilm.get_sentence_embedding_dimension()}")
print(f"e5-base dim: {e5.get_sentence_embedding_dimension()}")
print(f"BGE-M3 dim: {bge.get_sentence_embedding_dimension()}")

Using device: mps

Loading MiniLM (~500 MB)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6941.04it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loading e5-base (~1 GB)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7419.85it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loading BGE-M3 (~2.3 GB)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 60255.46it/s]



MiniLM dim: 384
e5-base dim: 768
BGE-M3 dim: 1024


/var/folders/xg/ply3c5550g1f_bg2ytt7chrw0000gn/T/ipykernel_35995/1132314349.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"\nMiniLM dim: {minilm.get_sentence_embedding_dimension()}")
/var/folders/xg/ply3c5550g1f_bg2ytt7chrw0000gn/T/ipykernel_35995/1132314349.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"e5-base dim: {e5.get_sentence_embedding_dimension()}")
/var/folders/xg/ply3c5550g1f_bg2ytt7chrw0000gn/T/ipykernel_35995/1132314349.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"BGE-M3 dim: {bge.get_sentence_embedding_dimension()}")


## 4. Embed the sample with each model — both raw and LaTeX-normalized

In [19]:
import numpy as np

raw_texts = [r["search_text"] for r in sample]
norm_texts = [normalize_latex(t) for t in raw_texts]

print("=== MiniLM ===")
minilm_raw_vecs  = minilm.encode(raw_texts,  normalize_embeddings=True, batch_size=64, show_progress_bar=True)
minilm_norm_vecs = minilm.encode(norm_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

print("\n=== e5-base (with passage: prefix) ===")
e5_raw_vecs  = e5.encode([f"passage: {t}" for t in raw_texts],  normalize_embeddings=True, batch_size=32, show_progress_bar=True)
e5_norm_vecs = e5.encode([f"passage: {t}" for t in norm_texts], normalize_embeddings=True, batch_size=32, show_progress_bar=True)

print("\n=== BGE-M3 ===")
bge_raw_vecs  = bge.encode(raw_texts,  normalize_embeddings=True, batch_size=16, show_progress_bar=True)
bge_norm_vecs = bge.encode(norm_texts, normalize_embeddings=True, batch_size=16, show_progress_bar=True)

print("\nAll embeddings ready.")

=== MiniLM ===


Batches: 100%|██████████| 5/5 [00:00<00:00,  7.46it/s]



=== e5-base (with passage: prefix) ===


Batches: 100%|██████████| 10/10 [00:02<00:00,  3.81it/s]



=== BGE-M3 ===


Batches: 100%|██████████| 19/19 [00:08<00:00,  2.25it/s]


All embeddings ready.


## 5. Retrieval helpers

In [20]:
# Single registry: each entry is (label, model_object, vecs, query_prefix, query_uses_norm)
MODELS = {
    "minilm-raw":  ("MiniLM (raw)",          minilm, minilm_raw_vecs,  "",          False),
    "minilm-norm": ("MiniLM (LaTeX-norm)",   minilm, minilm_norm_vecs, "",          True),
    "e5-raw":      ("e5-base (raw)",         e5,     e5_raw_vecs,      "query: ",   False),
    "e5-norm":     ("e5-base (LaTeX-norm)",  e5,     e5_norm_vecs,     "query: ",   True),
    "bge-raw":     ("BGE-M3 (raw)",          bge,    bge_raw_vecs,     "",          False),
    "bge-norm":    ("BGE-M3 (LaTeX-norm)",   bge,    bge_norm_vecs,    "",          True),
}

def retrieve(query: str, key: str, top_k: int = 3):
    label, model, vecs, prefix, uses_norm = MODELS[key]
    q = normalize_latex(query) if uses_norm else query
    q_vec = model.encode(prefix + q, normalize_embeddings=True)
    scores = vecs @ q_vec
    top = np.argsort(-scores)[:top_k]
    return [(sample[i], float(scores[i])) for i in top]

def compare(query: str, label: str, keys: list[str], top_k: int = 3) -> None:
    print("=" * 100)
    print(f"[{label}] Query: {query!r}")
    print("=" * 100)
    for key in keys:
        model_label = MODELS[key][0]
        print(f"\n── {model_label} ──")
        for row, score in retrieve(query, key, top_k):
            subj = '/'.join(row['subjects'] or ['(no subj)'])[:25]
            print(f"  [{score:+.3f}] {row['language']:>2s} | {subj:<25s} | {row['question_text'][:120]}")

## 6. Comparisons

### 6a. Three models — language-learning queries (no math, LaTeX-norm shouldn't matter)

In [21]:
MODELS_RAW_3WAY = ["minilm-raw", "e5-raw", "bge-raw"]

compare("past tense verbs conjugation", "EN grammar",       MODELS_RAW_3WAY)
compare("vocabulary about food and cooking", "EN vocab",    MODELS_RAW_3WAY)
compare("النحو والصرف في اللغة العربية", "AR grammar",      MODELS_RAW_3WAY)

[EN grammar] Query: 'past tense verbs conjugation'

── MiniLM (raw) ──
  [+0.649] en | (no subj)                 | Choose the correct statement about forming the present perfect tense:
  [+0.539] en | ENGLISH                   | Question 4: How do we form the passive?
  [+0.528] en | ENGLISH                   | She __________(to ride) her bike to school every day last year.

── e5-base (raw) ──
  [+0.851] en | (no subj)                 | Choose the correct statement about forming the present perfect tense:
  [+0.842] en | ENGLISH                   | Question 4: How do we form the passive?
  [+0.836] en | ENGLISH                   | She __________(to ride) her bike to school every day last year.

── BGE-M3 (raw) ──
  [+0.564] en | (no subj)                 | Choose the correct statement about forming the present perfect tense:
  [+0.547] en | ENGLISH                   | Question 4: How do we form the passive?
  [+0.546] en | ENGLISH                   | Task 1: Pick the right option. The

### 6b. Three models — math queries (where LaTeX matters)

In [22]:
compare("dérivée d'une fonction",      "FR math — derivatives",  MODELS_RAW_3WAY)
compare("primitives et intégrales",    "FR math — integrals",    MODELS_RAW_3WAY)
compare("loi de Newton force et mouvement", "FR physics",        MODELS_RAW_3WAY)
compare("الدوال والمعادلات",            "AR math — functions",    MODELS_RAW_3WAY)

[FR math — derivatives] Query: "dérivée d'une fonction"

── MiniLM (raw) ──
  [+0.647] en | MATHEMATICS               | La fonction \(f\left(x\right)=\frac{a}{x},\ a\ne0\) est définie sur : a) \(\mathbb{R}\ \) b) \(\mathbb{R}+\) c) \(\mathb
  [+0.629] fr | (no subj)                 | Soit \(f\) une fonction définie sur \(\mathbb{R}\) tel que pour tout \(x\in\mathbb{R}\) , \(f(-x)+3f(x)=4x^3+2x\) alors 
  [+0.598] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ strictement\ positive\ \) \(sur\ \mathbb{R}\ tel\ que\ f\left(0\right)=1et\ f'\l

── e5-base (raw) ──
  [+0.852] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ strictement\ positive\ \) \(sur\ \mathbb{R}\ tel\ que\ f\left(0\right)=1et\ f'\l
  [+0.843] fr | MATHEMATICS               | \(La\ fonction\ défine\ sur\ \mathbb{R}\ par\ f\left(x\right)=\sqrt{4x^2+1}\) \(est\ dérivable\ sur\ et\ pour\ tout\ x\ 
  [+0.839] fr | MATHEMATICS               | \(u\ et\ v\ sont\ deux\ fonctions\

### 6c. LaTeX normalization effect on MiniLM (math queries)

Same model, same query — only difference is whether we normalized LaTeX before embedding.

In [23]:
MINILM_LATEX_AB = ["minilm-raw", "minilm-norm"]

compare("dérivée d'une fonction",   "FR math", MINILM_LATEX_AB)
compare("primitives et intégrales", "FR math", MINILM_LATEX_AB)
compare("الدوال والمعادلات",         "AR math", MINILM_LATEX_AB)

[FR math] Query: "dérivée d'une fonction"

── MiniLM (raw) ──
  [+0.647] en | MATHEMATICS               | La fonction \(f\left(x\right)=\frac{a}{x},\ a\ne0\) est définie sur : a) \(\mathbb{R}\ \) b) \(\mathbb{R}+\) c) \(\mathb
  [+0.629] fr | (no subj)                 | Soit \(f\) une fonction définie sur \(\mathbb{R}\) tel que pour tout \(x\in\mathbb{R}\) , \(f(-x)+3f(x)=4x^3+2x\) alors 
  [+0.598] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ strictement\ positive\ \) \(sur\ \mathbb{R}\ tel\ que\ f\left(0\right)=1et\ f'\l

── MiniLM (LaTeX-norm) ──
  [+0.647] en | MATHEMATICS               | La fonction \(f\left(x\right)=\frac{a}{x},\ a\ne0\) est définie sur : a) \(\mathbb{R}\ \) b) \(\mathbb{R}+\) c) \(\mathb
  [+0.629] fr | (no subj)                 | Soit \(f\) une fonction définie sur \(\mathbb{R}\) tel que pour tout \(x\in\mathbb{R}\) , \(f(-x)+3f(x)=4x^3+2x\) alors 
  [+0.598] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ st

### 6d. LaTeX normalization effect on BGE-M3 (math queries)

Same A/B, but with the strongest model. Does normalization still matter when the model is good?

In [24]:
BGE_LATEX_AB = ["bge-raw", "bge-norm"]

compare("dérivée d'une fonction",   "FR math", BGE_LATEX_AB)
compare("primitives et intégrales", "FR math", BGE_LATEX_AB)
compare("الدوال والمعادلات",         "AR math", BGE_LATEX_AB)

[FR math] Query: "dérivée d'une fonction"

── BGE-M3 (raw) ──
  [+0.669] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ strictement\ positive\ \) \(sur\ \mathbb{R}\ tel\ que\ f\left(0\right)=1et\ f'\l
  [+0.635] fr | (no subj)                 | La courbe (C) ci dessous représente une fonction f définie et dérivable sur \(]0;+\infty\) [ et \(A\) la partie du plan 
  [+0.619] fr | MATHEMATICS               | \(La\ fonction\ défine\ sur\ \mathbb{R}\ par\ f\left(x\right)=\sqrt{4x^2+1}\) \(est\ dérivable\ sur\ et\ pour\ tout\ x\ 

── BGE-M3 (LaTeX-norm) ──
  [+0.669] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ strictement\ positive\ \) \(sur\ \mathbb{R}\ tel\ que\ f\left(0\right)=1et\ f'\l
  [+0.635] fr | (no subj)                 | La courbe (C) ci dessous représente une fonction f définie et dérivable sur \(]0;+\infty\) [ et \(A\) la partie du plan 
  [+0.619] fr | MATHEMATICS               | \(La\ fonction\ défine\ sur\ \mathbb{R}\ p

### 6e. Cross-lingual queries — three models, all raw

English query about a topic that mostly lives in French/Arabic data. A good multilingual embedder should still find the right rows.

In [25]:
compare("function derivative mathematics",  "CROSS-LINGUAL math",     MODELS_RAW_3WAY)
compare("chemical reaction equilibrium",    "CROSS-LINGUAL chemistry", MODELS_RAW_3WAY)
compare("physics newton force",             "CROSS-LINGUAL physics",  MODELS_RAW_3WAY)

[CROSS-LINGUAL math] Query: 'function derivative mathematics'

── MiniLM (raw) ──
  [+0.695] en | MATHEMATICS               | La fonction \(f\left(x\right)=\frac{a}{x},\ a\ne0\) est définie sur : a) \(\mathbb{R}\ \) b) \(\mathbb{R}+\) c) \(\mathb
  [+0.683] fr | (no subj)                 | La courbe (C) ci dessous représente une fonction f définie et dérivable sur \(]0;+\infty\) [ et \(A\) la partie du plan 
  [+0.666] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ strictement\ positive\ \) \(sur\ \mathbb{R}\ tel\ que\ f\left(0\right)=1et\ f'\l

── e5-base (raw) ──
  [+0.866] fr | MATHEMATICS               | \(La\ fonction\ défine\ sur\ \mathbb{R}\ par\ f\left(x\right)=\sqrt{4x^2+1}\) \(est\ dérivable\ sur\ et\ pour\ tout\ x\ 
  [+0.864] fr | MATHEMATICS               | \(Soit\ f\ la\ fonction\ dérivable\ et\ strictement\ positive\ \) \(sur\ \mathbb{R}\ tel\ que\ f\left(0\right)=1et\ f'\l
  [+0.852] fr | MATHEMATICS               | \(u\ et\ v\ sont\ deux\ fonc

## 7. Decision worksheet (fill in after running)

Pick the winning model + LaTeX setting based on what you saw above:

**Model:** which one returned the most relevant rows on YOUR data?
- [ ] MiniLM is good enough → cheapest option, smallest infra
- [ ] e5-base wins → 1 GB, prefix dance, decent
- [ ] BGE-M3 wins → 2.3 GB, no prefix, strongest

**LaTeX normalization:** is it clearly worth it?
- [ ] Yes — math retrieval visibly improved → keep `normalize_latex: true` in pipeline.yaml
- [ ] No clear difference → flip to `false`, save the cleanup cost

**Notes (paste back to me if helpful):**

_..._